# Document Q&A — LangChain বাংলা টিউটোরিয়াল

এই নোটবুকে আমরা শিখব কীভাবে PDF document থেকে তথ্য বের করে সেই তথ্যের উপর ভিত্তি করে প্রশ্নের উত্তর দেওয়া যায়। এই পদ্ধতিকে বলা হয় **RAG (Retrieval-Augmented Generation)** — মানে, প্রথমে প্রাসঙ্গিক তথ্য খুঁজে আনো, তারপর সেটা দিয়ে উত্তর তৈরি করো।

| ধাপ | বিষয় | বিবরণ |
|-----|-------|-------|
| ১ | Document Loading | PDF ফাইল লোড করা |
| ২ | Text Splitting | বড় text-কে ছোট chunk-এ ভাগ করা |
| ৩ | Embedding | text-কে vector-এ রূপান্তর করা |
| ৪ | Vector Store (FAISS) | vector গুলো সংরক্ষণ ও অনুসন্ধান করা |
| ৫ | Retriever | প্রশ্নের সাথে মিলিয়ে chunk খোঁজা |
| ৬ | RAG Chain | সব কিছু একসাথে যুক্ত করে উত্তর দেওয়া |

## ধাপ ১ — PDF Document লোড করা

প্রথমে আমাদের `../documents` ফোল্ডারে থাকা সব PDF ফাইল লোড করতে হবে।

এখানে দুটো জিনিস ব্যবহার হচ্ছে:
- **`DirectoryLoader`** — পুরো ফোল্ডার স্ক্যান করে সব ফাইল খোঁজে
- **`PyPDFLoader`** — প্রতিটি PDF ফাইল পড়ে text বের করে

শেষে `documents` variable-এ একটি list থাকবে, যেখানে প্রতিটি PDF-এর প্রতিটি page আলাদা `Document` object হিসেবে থাকবে।

In [1]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# PDF files যেই folder এ আছে সেই path
pdf_path = "../documents"

# সব PDF load করবে
loader = DirectoryLoader(
    path=pdf_path,
    glob="*.pdf",
    loader_cls=PyPDFLoader,
)

documents = loader.load()

print(f"✅ Total documents loaded: {len(documents)}")
print(documents[0].page_content[:500])

/Users/tappware/Desktop/Langchain-Github/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Total documents loaded: 62
AgenticRAG: Agentic Retrieval for Enterprise Knowledge Bases
Susheel Suresh*†, Hazel Mak∗‡, Shangpo Chou, Fred Kroon, Sahil Bhatnagar
Microsoft Corporation
Abstract
We presentAgenticRAG, a practical agen-
tic harness for retrieval and analysis over en-
terprise knowledge bases. Standard RAG
pipelines place significant burden of grounding
on the search stack, constraining the language
model to a fixed candidate set chosen deep in
the retrieval process. Our approach reduces
this overdependence by 


## ধাপ ২ — Text Splitting (Chunking)

LLM-এর একটি সীমাবদ্ধতা আছে — সে একসাথে অনেক বড় text পড়তে পারে না (context window সীমিত)। তাই বড় document গুলোকে ছোট ছোট **chunk**-এ ভাগ করতে হয়।

**`RecursiveCharacterTextSplitter`** কীভাবে কাজ করে:
- প্রথমে `\n\n` (প্যারাগ্রাফ) দিয়ে ভাগ করার চেষ্টা করে
- না হলে `\n` (লাইন) দিয়ে ভাগ করে
- তারপর `.` (বাক্য) দিয়ে, এবং শেষে space দিয়ে

**`chunk_overlap`** রাখার কারণ: একটি chunk-এর শেষের কিছু অংশ পরের chunk-এর শুরুতে রাখা হয়, যাতে কোনো তথ্য মাঝখানে কেটে না যায়।

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,      # প্রতিটি chunk সর্বোচ্চ ২০০০ character
    chunk_overlap=300,    # পাশাপাশি chunk-এ ৩০০ character overlap থাকবে
    separators=["\n\n", "\n", ".", " "],  # এই ক্রমে ভাগ করার চেষ্টা করবে
)

chunks = splitter.split_documents(documents)

print(f"Original docs:    {len(documents)}")
print(f"After Splitting:  {len(chunks)} chunks")

Original docs:    62
After Splitting:  160 chunks


### একটি Chunk কেমন দেখতে?

নিচে প্রথম chunk-টি দেখা যাক। প্রতিটি chunk-এ দুটো অংশ থাকে:
- **`page_content`** — মূল text
- **`metadata`** — কোন PDF-এর কোন page থেকে এসেছে সেই তথ্য

In [3]:
chunks[0]

Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Susheel Suresh; Hazel Mak; Shangpo Chou; Fred Kroon; Sahil Bhatnagar', 'doi': 'https://doi.org/10.48550/arXiv.2605.05538', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'AgenticRAG: Agentic Retrieval for Enterprise Knowledge Bases', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2605.05538v1', 'source': '../documents/2605.05538v1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='AgenticRAG: Agentic Retrieval for Enterprise Knowledge Bases\nSusheel Suresh*†, Hazel Mak∗‡, Shangpo Chou, Fred Kroon, Sahil Bhatnagar\nMicrosoft Corporation\nAbstract\nWe presentAgenticRAG, a practical agen-\ntic harness for retrieval and analysis over en-\nterprise knowledge bases. Standard RAG\npipelines place significant 

## ধাপ ৩ — Embedding Model লোড করা

**Embedding** হলো text-কে সংখ্যার তালিকায় (vector) রূপান্তর করার প্রক্রিয়া। এই vector-গুলো দিয়ে দুটো text-এর মধ্যে অর্থগত মিল (semantic similarity) বোঝা যায়।

উদাহরণ:
- "কুকুর" এবং "বিড়াল" — এরা কাছাকাছি vector পাবে (দুটোই প্রাণী)
- "কুকুর" এবং "মোটরগাড়ি" — এরা দূরের vector পাবে

এখানে **`all-MiniLM-L6-v2`** model ব্যবহার করা হচ্ছে — এটি HuggingFace-এর একটি জনপ্রিয় ছোট embedding model যা:
- সম্পূর্ণ বিনামূল্যে
- locally (তোমার কম্পিউটারে) চলে
- কোনো API key লাগে না

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

# HuggingFace embedding model লোড করো (সম্পূর্ণ বিনামূল্যে, locally চলে)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embedding model লোড হয়েছে!")
print("Model: all-MiniLM-L6-v2")

/Users/tappware/Desktop/Langchain-Github/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Embedding model লোড হয়েছে!
Model: all-MiniLM-L6-v2


## ধাপ ৪ — FAISS Vector Store তৈরি করা

**FAISS (Facebook AI Similarity Search)** হলো একটি library যা দ্রুত vector similarity search করতে পারে।

এই ধাপে যা হচ্ছে:
1. প্রতিটি chunk-এর text নিয়ে embedding model দিয়ে vector তৈরি হচ্ছে
2. সেই vector গুলো FAISS-এর database-এ সংরক্ষণ হচ্ছে
3. পরে প্রশ্ন করলে, প্রশ্নের vector-এর সাথে মিলিয়ে সবচেয়ে কাছের chunk খুঁজে বের করা হবে

```
chunk text  →  embedding model  →  vector  →  FAISS store
```

In [5]:
from langchain_community.vectorstores import FAISS

# সব chunk-এর embedding তৈরি করে FAISS store-এ রাখো
vector_store = FAISS.from_documents(chunks, embeddings)

print("✅ FAISS Vector Store তৈরি হয়েছে!")
print(f"মোট documents: {len(documents)}")

✅ FAISS Vector Store তৈরি হয়েছে!
মোট documents: 62


## ধাপ ৫ — Retriever তৈরি করা

**Retriever** হলো একটি interface যা প্রশ্ন দিলে সবচেয়ে প্রাসঙ্গিক chunk গুলো ফিরিয়ে দেয়।

এখানে:
- `search_type="similarity"` — cosine similarity দিয়ে কাছের chunk খোঁজে
- `k=3` — সবচেয়ে কাছের ৩টি chunk ফিরিয়ে দেবে

পরে এই retriever-কে RAG chain-এর সাথে যুক্ত করা হবে।

In [6]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # সবচেয়ে কাছের ৩টি chunk আনবে
)

## ধাপ ৬ — LLM সেটআপ করা

এখানে Google-এর **Gemini** model ব্যবহার করা হচ্ছে উত্তর তৈরি করতে।

`.env` ফাইলে `GOOGLE_API_KEY` রাখতে হবে:
```
GOOGLE_API_KEY=তোমার_api_key_এখানে
```

`python-dotenv` দিয়ে সেই key লোড করা হচ্ছে যাতে code-এর মধ্যে key সরাসরি লিখতে না হয়।

In [7]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
)

## ধাপ ৭ — RAG Chain তৈরি ও প্রশ্ন করা

এখন সব কিছু একসাথে যুক্ত করে পূর্ণ RAG chain তৈরি হবে।

**Chain-এর flow:**
```
প্রশ্ন
  ↓
Retriever  →  সবচেয়ে কাছের ৩টি chunk খোঁজে
  ↓
Prompt     →  context + question একসাথে সাজায়
  ↓
LLM        →  উত্তর তৈরি করে
  ↓
Parser     →  string হিসেবে output দেয়
```

**গুরুত্বপূর্ণ বিষয়:** LLM-এর prompt-এ শুধু মূল text (context) পাঠানো হচ্ছে — source বা metadata পাঠানো হচ্ছে না। উত্তর তৈরির পর, retrieved chunk গুলোর metadata থেকে আলাদাভাবে source বের করে print করা হবে।

In [13]:
# RAG Prompt — শুধু context ও question; source metadata এখানে নেই
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant. Answer only based on the provided context.
If the answer is not in the context, say 'I don't know'.

Context:
{context}""",
    ),
    ("human", "{question}"),
])


def format_docs(docs):
    """Document গুলোর শুধু text অংশ একসাথে জোড়া দাও।"""
    return "\n\n".join(doc.page_content for doc in docs)


def get_sources(docs):
    """Retrieved chunk গুলোর source ও page number বের করো।"""
    seen = set()
    sources = []
    for doc in docs:
        source = doc.metadata.get("source", "অজানা")
        page   = doc.metadata.get("page", 0)  # 0-indexed, তাই +1 করা হবে
        content= doc.page_content
        key    = (source, page)
        if key not in seen:
            seen.add(key)
            sources.append({"source": source, "page": int(page) + 1 , "content": content})
    return sources


def ask(question: str) -> None:
    """প্রশ্ন করো এবং উত্তর ও source আলাদাভাবে print করো।"""

    # ১. Retriever দিয়ে প্রাসঙ্গিক chunk গুলো আনো
    retrieved_docs = retriever.invoke(question)

    # ২. RAG chain চালাও — শুধু text context পাঠাও, metadata নয়
    rag_chain = (
        {"context": lambda _: format_docs(retrieved_docs), "question": RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    )
    answer = rag_chain.invoke(question)

    # ৩. উত্তর print করো
    print(f"প্রশ্ন: {question}")
    print(f"\nউত্তর:\n{answer}")

    # ৪. Source আলাদাভাবে print করো
    sources = get_sources(retrieved_docs)
    print("\n" + "-" * 42)
    print("সূত্র (Sources):")
    print("-" * 42)
    for i, src in enumerate(sources, start=1):
        print(f"  [{i}] {src['source']} - Page {src['page']}")
        print()
        print(f"Content: \n {src['content'][:100]} ...")
    print("-" * 42)


# প্রশ্ন করো
ask("What is TituLLMs?")

প্রশ্ন: What is TituLLMs?

উত্তর:
TituLLM is a language model that has variants like TituLLM-3b and TituLLM-1b. It demonstrates strong performance in commonsense reasoning tasks such as CSQA, OBQA, and PIQA, suggesting good reasoning capabilities, especially for Bangla Language-specific reasoning. A significant feature of TituLLM is its extended tokenizer, which processes Bangla text into word or sub-word levels, delivering more meaningful tokens compared to Llama's character and byte-level splitting. This tokenization strategy allows TituLLM to perform better with smaller datasets and ensures low latency during inference.

------------------------------------------
সূত্র (Sources):
------------------------------------------
  [1] ../documents/2502.11187v3.pdf - Page 8

Content: 
 Figure 6: Example of tokenization of Llama and Tit-
uLLM tokenizers.
CSQA, OBQA, and PIQA:Commonsens ...
  [2] ../documents/2502.11187v3.pdf - Page 4

Content: 
 We have also evaluated our model using SQM
(La

In [14]:
ask("What is main purpose of 'Attention Is Not All You Need for Diffraction'?")

প্রশ্ন: What is main purpose of 'Attention Is Not All You Need for Diffraction'?

উত্তর:
I don't know.

------------------------------------------
সূত্র (Sources):
------------------------------------------
  [1] ../documents/2604.23811v1.pdf - Page 17

Content: 
 17
S8. CONFUSION MATRIX
FIG. S3. Confusion matrix for the vision transformer on the synthetic balanc ...
  [2] ../documents/2604.23811v1.pdf - Page 11

Content: 
 tational resources were also provided through ACCESS
allocation PHY250007, “Applications of AI to Di ...
  [3] ../documents/2604.23811v1.pdf - Page 1

Content: 
 by an expert crystallographer—a process poorly suited
to the massive data volumes produced by modern ...
------------------------------------------


---

## 📝 সারসংক্ষেপ

> **মূল কথা:** RAG pipeline-এ তিনটি বড় ধাপ আছে —
> ১. **Index করা:** PDF লোড → chunk করা → embedding → FAISS-এ সংরক্ষণ
> ২. **Retrieve করা:** প্রশ্নের সাথে মিলিয়ে সবচেয়ে কাছের chunk খোঁজা
> ৩. **Generate করা:** সেই chunk-গুলো context হিসেবে দিয়ে LLM দিয়ে উত্তর তৈরি করা

| বিষয় | ব্যবহৃত tool |
|-------|-------------|
| PDF লোড | `DirectoryLoader` + `PyPDFLoader` |
| Chunking | `RecursiveCharacterTextSplitter` |
| Embedding | `HuggingFaceEmbeddings (all-MiniLM-L6-v2)` |
| Vector Store | `FAISS` |
| LLM | `ChatGoogleGenerativeAI (Gemini)` |
| Source tracking | metadata থেকে আলাদাভাবে বের করা |